# 02 — Baseline TF-IDF Model Benchmark

**Goal:** Establish baseline performance with current TF-IDF + LogisticRegression model

**Outputs:**
- Baseline accuracy, precision, recall, F1
- Confusion matrix
- Feature importance analysis
- Benchmark to beat with transformer

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

## 1. Load Data

In [ ]:
# Load cleaned dataset from notebook 01
df = pd.read_csv('../backend/training/dataset_clean.csv')
print(f"Dataset size: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")

X = df['text']
y = df['label']

## 2. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

## 3. TF-IDF Vectorization

In [ ]:
# Match your current production settings
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words='english'
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Feature matrix shape: {X_train_vec.shape}")

## 4. Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
model.fit(X_train_vec, y_train)

print("Training complete!")

## 5. Evaluation

In [ ]:
# Predictions
y_pred = model.predict(X_test_vec)
y_pred_proba = model.predict_proba(X_test_vec)[:, 1]

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nROC-AUC Score: {roc_auc:.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real', 'Fake'], 
            yticklabels=['Real', 'Fake'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix - TF-IDF Baseline')
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - TF-IDF Baseline')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Feature Importance

In [ ]:
# Top features for fake news
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

# Top 20 features indicating FAKE
top_fake_idx = np.argsort(coefficients)[-20:][::-1]
top_fake_features = [(feature_names[i], coefficients[i]) for i in top_fake_idx]

# Top 20 features indicating REAL
top_real_idx = np.argsort(coefficients)[:20]
top_real_features = [(feature_names[i], coefficients[i]) for i in top_real_idx]

print("Top 20 features indicating FAKE news:")
for feat, coef in top_fake_features:
    print(f"  {feat}: {coef:.4f}")

print("\nTop 20 features indicating REAL news:")
for feat, coef in top_real_features:
    print(f"  {feat}: {coef:.4f}")

## 7. Cross-Validation

In [ ]:
# 5-fold cross-validation
X_all_vec = vectorizer.fit_transform(X)
cv_scores = cross_val_score(model, X_all_vec, y, cv=5, scoring='f1')

print(f"Cross-validation F1 scores: {cv_scores}")
print(f"Mean F1: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## 8. Save Baseline Results

In [ ]:
# Save model and vectorizer
joblib.dump(model, '../backend/data/baseline_model.joblib')
joblib.dump(vectorizer, '../backend/data/baseline_vectorizer.joblib')

# Save metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

baseline_metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc,
    'cv_f1_mean': cv_scores.mean(),
    'cv_f1_std': cv_scores.std()
}

import json
with open('../backend/data/baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

print("\nBaseline metrics saved!")
print(json.dumps(baseline_metrics, indent=2))

## Summary

**Baseline Performance (TF-IDF + LogisticRegression):**
- This is the benchmark to beat with the transformer model
- Target: 95%+ accuracy (vs current ~90%)
- Next: Train DeBERTa in notebook 03